# Paralelismo de Memoria Compartida: `multiprocessing` sobre el Escáner de Parámetros

*Métodos Computacionales Modernos para la Física — Módulo II: Fundamentos de Alto Desempeño*

El cuaderno anterior (`jit_numba.ipynb`) cerró con una cifra que sesión tras
sesión se ha ido reduciendo pero nunca ha cambiado de naturaleza: el
escaneo 5D de referencia ($20^5=3{,}200{,}000$ puntos) pasó de 43 días (quad
original) a ~4 días (NumPy vectorizado, Sesión 6) a poco menos de 3 días
(Numba, Sesión 7) — **siempre sobre un solo núcleo**. Cada punto del escaneo
es independiente de todos los demás: no comparte estado, no depende de un
punto vecino, no impone ningún orden de evaluación. Es, en la terminología
estándar, **"vergonzosamente paralelo"** — el mejor caso posible para repartir
entre varios núcleos, porque no hace falta ninguna comunicación entre ellos
mientras dura el cómputo.

Hoy repartimos ese escaneo entre los núcleos de una sola máquina con el
módulo `multiprocessing` de la biblioteca estándar, y cerramos, con números
reales, la pregunta que la Sesión 7 dejó pendiente en su Sección 11: ¿qué le
pasa al binario ya compilado por Numba cuando el escaneo se reparte entre
varios procesos?

In [1]:
import sys, math, time, subprocess, textwrap
import multiprocessing as mp
import numpy as np
import numba
from numba import njit
from scipy.integrate import solve_ivp
from scipy.special import kve, kn

print(f"Python  {sys.version.split()[0]}")
print(f"NumPy   {np.__version__}")
print(f"Numba   {numba.__version__}")
print(f"Núcleos lógicos (mp.cpu_count()): {mp.cpu_count()}")
print(f"Método de arranque por defecto:   {mp.get_start_method()}")

resultados = {}

Python  3.14.0
NumPy   2.3.4
Numba   0.67.0
Núcleos lógicos (mp.cpu_count()): 12
Método de arranque por defecto:   forkserver


## 1. La pregunta de la Sesión 7: ¿qué pasa con el binario compilado?

`multiprocessing.Pool` puede crear procesos hijos de tres maneras
(`mp.get_context('fork'|'spawn'|'forkserver')`), y la diferencia entre ellas
es precisamente la que la Sesión 7 dejó como pregunta abierta:

- **`fork`** (default histórico en Linux): el proceso hijo es una copia
  *copy-on-write* de la memoria del proceso padre en el instante exacto de
  crearlo. Si el padre ya importó Numba y ya compiló `thermal_average`, el
  hijo hereda ese binario ya compilado en memoria — sin volver a importar
  nada, sin volver a compilar nada.
- **`spawn`** (default en macOS y Windows) y **`forkserver`** (default en
  Linux desde Python 3.14, y el que este mismo intérprete reportó arriba):
  el hijo arranca como un intérprete de Python genuinamente nuevo, que debe
  volver a importar el módulo principal — incluyendo `import numba` y la
  definición de cada función `@njit` — antes de poder ejecutar una sola tarea.

No hace falta especular: se puede medir directamente el costo de un intérprete
nuevo, lanzando procesos reales con `subprocess` (no con `multiprocessing`,
precisamente para no heredar nada del proceso de este cuaderno).

In [2]:
script = textwrap.dedent('''
    import time
    t0 = time.perf_counter()
    import numba
    from numba import njit
    t1 = time.perf_counter()
    print(f"import numba: {1e3*(t1-t0):.1f} ms")

    @njit(cache=True)
    def kernel(x):
        s = 0.0
        for i in range(x.shape[0]):
            s += x[i]**2
        return s

    import numpy as np
    x = np.random.default_rng(0).normal(size=100_000)
    t0 = time.perf_counter()
    kernel(x)
    t1 = time.perf_counter()
    print(f"primera llamada (compilar o leer cache de disco): {1e3*(t1-t0):.1f} ms")
''')
with open('_cold_worker.py', 'w') as f:
    f.write(script)

import shutil
shutil.rmtree('__pycache__', ignore_errors=True)

print("--- Corrida 1: sin cache de disco todavia ---")
r1 = subprocess.run([sys.executable, '_cold_worker.py'], capture_output=True, text=True)
print(r1.stdout)

print("--- Corrida 2: proceso nuevo otra vez, pero el cache de disco de la Corrida 1 ya existe ---")
r2 = subprocess.run([sys.executable, '_cold_worker.py'], capture_output=True, text=True)
print(r2.stdout)

--- Corrida 1: sin cache de disco todavia ---


import numba: 680.3 ms
primera llamada (compilar o leer cache de disco): 1131.5 ms

--- Corrida 2: proceso nuevo otra vez, pero el cache de disco de la Corrida 1 ya existe ---


import numba: 690.8 ms
primera llamada (compilar o leer cache de disco): 702.4 ms



El resultado se lee en dos partes. `import numba` en sí mismo —antes de
compilar absolutamente nada— ya cuesta varios cientos de milisegundos en esta
máquina: es el costo fijo, por-proceso, de que Numba inicialice su
maquinaria de tipado y LLVM, exactamente el costo que la Sesión 7 identificó
pero no aisló de la compilación misma. La segunda línea de cada corrida sí
distingue el efecto de `cache=True`: con el cache de disco ya presente
(Corrida 2), la primera llamada es más rápida que en la Corrida 1 — ese es el
paso de compilación LLVM que `cache=True` evita, y es real — pero sigue
costando casi lo mismo que `import numba` por sí solo. **`cache=True` elimina
la recompilación; no elimina el costo de arrancar un intérprete de Python
nuevo y que Numba se inicialice dentro de él.** Eso es exactamente lo que le
pasa a cada proceso hijo bajo `spawn`/`forkserver`, y exactamente lo que
**no** le pasa a un hijo bajo `fork`, porque un hijo `fork` nunca vuelve a
ejecutar `import numba` — ya está en su memoria heredada, ya compilada.

## 2. El caso de estudio: `thermal_average` compilada (Sesión 7), reempaquetada

Antes de paralelizar hace falta algo que valga la pena paralelizar. Se
reconstruye aquí, sin cambios, el camino GL+`njit` de la Sesión 7 —el más
rápido de los dos caminos que esa sesión validó— como una función de nivel
de módulo: **esto no es cosmético**. `multiprocessing.Pool.map` necesita
poder serializar (`pickle`) tanto la función de trabajo como sus argumentos
para enviarlos a cada proceso hijo; un método ligado a una instancia
(`self.thermal_average`) o una función `lambda` con estado capturado
frecuentemente no se puede serializar así. Una función de nivel de módulo,
con argumentos explícitos, sí.

In [3]:
@njit(cache=True)
def _bessk1e(x):
    if x <= 2.0:
        t = (x/3.75)**2
        i1 = x*(0.5 + t*(0.87890594 + t*(0.51498869 + t*(0.15084934 + t*(0.02658733 + t*(0.00301532 + t*0.00032411))))))
        y = x*x/4.0
        k1 = (math.log(x/2.0)*i1) + (1.0/x)*(1.0 + y*(0.15443144 + y*(-0.67278579 +
             y*(-0.18156897 + y*(-0.01919402 + y*(-0.00110404 + y*(-0.00004686)))))))
        return k1 * math.exp(x)
    else:
        y = 2.0/x
        return (1.0/math.sqrt(x)) * (1.25331414 + y*(0.23498619 + y*(-0.0365562 +
               y*(0.01504268 + y*(-0.00780353 + y*(0.00325614 + y*(-0.00068245)))))))

@njit(cache=True)
def _bessk0e(x):
    if x <= 2.0:
        t = (x/3.75)**2
        i0 = 1.0 + t*(3.5156229 + t*(3.0899424 + t*(1.2067492 + t*(0.2659732 + t*(0.0360768 + t*0.0045813)))))
        y = x*x/4.0
        k0 = (-math.log(x/2.0)*i0) + (-0.57721566 + y*(0.42278420 + y*(0.23069756 +
             y*(0.0348859 + y*(0.00262698 + y*(0.0001075 + y*0.0000074))))))
        return k0 * math.exp(x)
    else:
        y = 2.0/x
        return (1.0/math.sqrt(x)) * (1.25331414 + y*(-0.07832358 + y*(0.02189568 +
               y*(-0.01062446 + y*(0.00587872 + y*(-0.0025154 + y*0.00053208))))))

@njit(cache=True)
def _bessk2e(x):
    # Sesion 7, Seccion 5: recurrencia K_{v+1} = K_{v-1} + (2v/z) K_v, valida en forma escalada
    return _bessk0e(x) + (2.0/x) * _bessk1e(x)

_N_NODES = 64
_GL_X, _GL_W = np.polynomial.legendre.leggauss(_N_NODES)

@njit(cache=True)
def _thermal_average_gl_native(m, T, sigma0, nodes, weights):
    s_min = 4.0 * m * m
    s_max = (2.0*m + 30.0*T)**2
    half = 0.5 * (s_max - s_min)
    mid  = 0.5 * (s_max + s_min)
    total = 0.0
    for i in range(nodes.shape[0]):
        s = half * nodes[i] + mid
        if s > s_min:
            sqrt_s = math.sqrt(s)
            arg = (sqrt_s - 2.0*m) / T
            exp_factor = math.exp(-arg) if arg < 100.0 else 0.0
            val = sigma0 * (s - s_min) * sqrt_s * _bessk1e(sqrt_s / T) * exp_factor
        else:
            val = 0.0
        total += weights[i] * val
    integral = half * total
    k2 = _bessk2e(m / T)
    return 0.0 if k2 <= 0.0 else integral / (8.0 * m**4 * T * k2 * k2)

# Validacion rapida contra el oraculo scipy (Sesion 7 ya hizo la validacion completa; aqui solo confirmamos que la copia es fiel)
xs_val = np.geomspace(0.001, 1000, 50)
err = max(abs(_bessk1e(x) - kve(1, x))/abs(kve(1, x)) for x in xs_val)
print(f"bessk1e, error relativo maximo vs scipy.special.kve: {err:.2e}")
assert err < 1e-6

M_PLANCK = 1.2209e19
RHO_CRIT_OVER_H2 = 1.0537e-5 * ((5.0677e13)**-3)
S_NOW = 2891.2 * ((5.0677e13)**-3)
G_STAR = 106.75  # relic.py: g_star, g_star_s constantes en este modelo

def y_eq(x, m):
    return (45.0/(4*np.pi**4)) * (2.0/G_STAR) * (x**2) * kn(2, x)

def calculate_omega_h2_fast(m, sigma0, x_start=1.0, x_end=1000.0):
    '''Version de nivel de modulo (picklable) de RelicSolver.calculate_omega_h2,
    usando la thermal_average GL+njit de la Sesion 7 en vez de scipy.integrate.quad.'''
    def rhs(x, logy):
        T = m/x
        h = 1.66*np.sqrt(G_STAR)*(T**2)/M_PLANCK
        s_ent = (2*np.pi**2/45)*G_STAR*(T**3)
        sigv = _thermal_average_gl_native(m, T, sigma0, _GL_X, _GL_W)
        y_eq_val = y_eq(x, m)
        if y_eq_val <= 0:
            return [-(s_ent*sigv/(h*x))*math.exp(logy[0])]
        diff = logy[0] - math.log(y_eq_val)
        if diff > 50:
            return [-(s_ent*sigv/(h*x))*math.exp(logy[0])]
        return [-(s_ent*sigv/(h*x))*2*y_eq_val*math.sinh(diff)]
    y0 = y_eq(x_start, m)
    sol = solve_ivp(rhs, [x_start, x_end], [math.log(y0)], method='Radau', rtol=1e-6, atol=1e-12)
    return m * S_NOW * math.exp(sol.y[0][-1]) / RHO_CRIT_OVER_H2

_thermal_average_gl_native(100.0, 5.0, 1e-9, _GL_X, _GL_W)  # calentamiento

t0 = time.perf_counter()
oh2_ref = calculate_omega_h2_fast(100.0, 1e-9)
t1 = time.perf_counter()
print(f"calculate_omega_h2_fast(m=100, sigma0=1e-9) = {oh2_ref:.6f}  ({1e3*(t1-t0):.1f} ms, ya compilado)")
assert abs(oh2_ref - 0.535064) / 0.535064 < 1e-3, "Deberia coincidir con la baseline de la Sesion 7"
resultados['un punto — solve() individual'] = t1 - t0

bessk1e, error relativo maximo vs scipy.special.kve: 1.65e-07


calculate_omega_h2_fast(m=100, sigma0=1e-9) = 0.535064  (310.5 ms, ya compilado)


El valor coincide, dentro de la tolerancia esperada, con `Omega h^2 =
0.535064` que la Sesión 7 reportó para el mismo punto — la portabilidad del
código entre sesiones queda validada antes de medir un solo tiempo de
escaneo.

## 3. El escáner, en serie

Rejilla de referencia para este cuaderno: $10\times10$ puntos en
$(m, \sigma_0)$ — deliberadamente más pequeña que el escaneo 5D completo,
para que la comparación serie/paralelo corra en un tiempo razonable de
clase; la extrapolación al escaneo completo llega en la Sección 5.

In [4]:
def scan_point(args):
    '''Funcion de nivel de modulo: la unica forma confiablemente picklable
    de pasarle trabajo a un Pool de multiprocessing.'''
    m, sigma0 = args
    return calculate_omega_h2_fast(m, sigma0)

m_grid = np.linspace(50.0, 150.0, 10)
sigma0_grid = np.geomspace(1e-10, 1e-8, 10)
grid = [(m, s) for m in m_grid for s in sigma0_grid]
print(f"Tamano de la rejilla: {len(grid)} puntos")

t0 = time.perf_counter()
resultados_serie = [scan_point(pt) for pt in grid]
t1 = time.perf_counter()
t_serie = t1 - t0
resultados[f'rejilla {len(grid)} puntos — serie (1 nucleo)'] = t_serie
print(f"Serie: {t_serie:.2f} s total, {1e3*t_serie/len(grid):.1f} ms/punto")

Tamano de la rejilla: 100 puntos


Serie: 28.05 s total, 280.5 ms/punto


## 4. El mismo escáner, con `multiprocessing.Pool` — y un obstáculo real

`Pool.map` reparte la lista de `grid` entre `processes` procesos hijo,
recolecta los resultados en el mismo orden en que se pidieron, y los junta de
vuelta en el proceso principal — el patrón estándar para un problema
vergonzosamente paralelo, sin escribir ninguna lógica de sincronización a
mano.

La primera versión de este cuaderno intentó exactamente eso, `mp.Pool(...)`
llamado directamente desde esta misma celda de kernel de Jupyter — y se
quedó colgada: 15 minutos y 47 segundos de reloj, de los cuales el propio
sistema operativo reportó menos de un minuto de tiempo de CPU real. Ese
desajuste (mucho tiempo de reloj, casi nada de CPU) es la firma de un
proceso **bloqueado esperando algo que nunca llega**, no de un cómputo lento.
Vale la pena diagnosticarlo en vez de simplemente evitarlo, porque es un
obstáculo real que cualquiera que use `multiprocessing` desde un notebook
eventualmente encuentra:

Un kernel de Jupyter (`ipykernel`) no es un intérprete de Python de un solo
hilo — corre su propio bucle de eventos asíncrono y varios hilos en segundo
plano (para la comunicación ZeroMQ con el frontend, entre otras cosas). Crear
un proceso hijo desde un proceso que ya tiene varios hilos activos es,
textualmente, la situación que POSIX documenta como insegura para `fork()` —
el hijo hereda sólo el hilo que llamó a `fork`, pero puede heredar bloqueos
(*locks*) que otros hilos del padre tenían tomados en el instante exacto de
la bifurcación, y esos bloqueos nunca se liberan en el hijo porque el hilo
que los liberaría no existe ahí. Es, de hecho, la misma razón de fondo por la
que este mismo intérprete (Sección 0) reportó `forkserver` como método de
arranque por defecto en vez del histórico `fork`: Python ha ido alejándose de
`fork` puro precisamente por este riesgo en procesos con hilos — y un kernel
de Jupyter es, casi por definición, un proceso con hilos.

La solución estándar, y la que se usa aquí, es la misma que ya empleó la
Sección 1: ejecutar el cómputo paralelo como un **script externo**, con
`subprocess`, fuera del proceso del kernel — exactamente como correría en
producción, sin ningún notebook de por medio.

In [5]:
scan_script = textwrap.dedent(r'''
    import sys, time, math
    import multiprocessing as mp
    import numpy as np
    from numba import njit
    from scipy.integrate import solve_ivp
    from scipy.special import kn

    @njit(cache=True)
    def _bessk1e(x):
        if x <= 2.0:
            t = (x/3.75)**2
            i1 = x*(0.5 + t*(0.87890594 + t*(0.51498869 + t*(0.15084934 + t*(0.02658733 + t*(0.00301532 + t*0.00032411))))))
            y = x*x/4.0
            k1 = (math.log(x/2.0)*i1) + (1.0/x)*(1.0 + y*(0.15443144 + y*(-0.67278579 +
                 y*(-0.18156897 + y*(-0.01919402 + y*(-0.00110404 + y*(-0.00004686)))))))
            return k1 * math.exp(x)
        else:
            y = 2.0/x
            return (1.0/math.sqrt(x)) * (1.25331414 + y*(0.23498619 + y*(-0.0365562 +
                   y*(0.01504268 + y*(-0.00780353 + y*(0.00325614 + y*(-0.00068245)))))))

    @njit(cache=True)
    def _bessk0e(x):
        if x <= 2.0:
            t = (x/3.75)**2
            i0 = 1.0 + t*(3.5156229 + t*(3.0899424 + t*(1.2067492 + t*(0.2659732 + t*(0.0360768 + t*0.0045813)))))
            y = x*x/4.0
            k0 = (-math.log(x/2.0)*i0) + (-0.57721566 + y*(0.42278420 + y*(0.23069756 +
                 y*(0.0348859 + y*(0.00262698 + y*(0.0001075 + y*0.0000074))))))
            return k0 * math.exp(x)
        else:
            y = 2.0/x
            return (1.0/math.sqrt(x)) * (1.25331414 + y*(-0.07832358 + y*(0.02189568 +
                   y*(-0.01062446 + y*(0.00587872 + y*(-0.0025154 + y*0.00053208))))))

    @njit(cache=True)
    def _bessk2e(x):
        return _bessk0e(x) + (2.0/x) * _bessk1e(x)

    _N_NODES = 64
    _GL_X, _GL_W = np.polynomial.legendre.leggauss(_N_NODES)

    @njit(cache=True)
    def _thermal_average_gl_native(m, T, sigma0, nodes, weights):
        s_min = 4.0 * m * m
        s_max = (2.0*m + 30.0*T)**2
        half = 0.5 * (s_max - s_min)
        mid  = 0.5 * (s_max + s_min)
        total = 0.0
        for i in range(nodes.shape[0]):
            s = half * nodes[i] + mid
            if s > s_min:
                sqrt_s = math.sqrt(s)
                arg = (sqrt_s - 2.0*m) / T
                exp_factor = math.exp(-arg) if arg < 100.0 else 0.0
                val = sigma0 * (s - s_min) * sqrt_s * _bessk1e(sqrt_s / T) * exp_factor
            else:
                val = 0.0
            total += weights[i] * val
        integral = half * total
        k2 = _bessk2e(m / T)
        return 0.0 if k2 <= 0.0 else integral / (8.0 * m**4 * T * k2 * k2)

    M_PLANCK = 1.2209e19
    RHO_CRIT_OVER_H2 = 1.0537e-5 * ((5.0677e13)**-3)
    S_NOW = 2891.2 * ((5.0677e13)**-3)
    G_STAR = 106.75

    def y_eq(x, m):
        return (45.0/(4*np.pi**4)) * (2.0/G_STAR) * (x**2) * kn(2, x)

    def calculate_omega_h2_fast(m, sigma0, x_start=1.0, x_end=1000.0):
        def rhs(x, logy):
            T = m/x
            h = 1.66*np.sqrt(G_STAR)*(T**2)/M_PLANCK
            s_ent = (2*np.pi**2/45)*G_STAR*(T**3)
            sigv = _thermal_average_gl_native(m, T, sigma0, _GL_X, _GL_W)
            y_eq_val = y_eq(x, m)
            if y_eq_val <= 0:
                return [-(s_ent*sigv/(h*x))*math.exp(logy[0])]
            diff = logy[0] - math.log(y_eq_val)
            if diff > 50:
                return [-(s_ent*sigv/(h*x))*math.exp(logy[0])]
            return [-(s_ent*sigv/(h*x))*2*y_eq_val*math.sinh(diff)]
        y0 = y_eq(x_start, m)
        sol = solve_ivp(rhs, [x_start, x_end], [math.log(y0)], method='Radau', rtol=1e-6, atol=1e-12)
        return m * S_NOW * math.exp(sol.y[0][-1]) / RHO_CRIT_OVER_H2

    def scan_point(args):
        m, sigma0 = args
        return calculate_omega_h2_fast(m, sigma0)

    if __name__ == "__main__":
        m_grid = np.linspace(50.0, 150.0, 10)
        sigma0_grid = np.geomspace(1e-10, 1e-8, 10)
        grid = [(m, s) for m in m_grid for s in sigma0_grid]

        _thermal_average_gl_native(100.0, 5.0, 1e-9, _GL_X, _GL_W)  # calentamiento

        t0 = time.perf_counter()
        resultados_serie = [scan_point(pt) for pt in grid]
        t1 = time.perf_counter()
        t_serie = t1 - t0
        print(f"RESULT|serie|{t_serie:.6f}")
        print(f"Serie: {t_serie:.2f} s total, {1e3*t_serie/len(grid):.1f} ms/punto", file=sys.stderr)

        for nproc in (2, 4, mp.cpu_count()):
            t0 = time.perf_counter()
            with mp.Pool(processes=nproc) as pool:
                resultados_par = pool.map(scan_point, grid)
            t1 = time.perf_counter()
            t_par = t1 - t0
            assert np.allclose(resultados_serie, resultados_par, rtol=1e-6)
            speedup = t_serie / t_par
            eficiencia = 100.0 * speedup / nproc
            print(f"RESULT|pool{nproc}|{t_par:.6f}")
            print(f"Pool({nproc:2d} procesos): {t_par:6.2f} s total, {1e3*t_par/len(grid):6.1f} ms/punto, "
                  f"aceleracion {speedup:4.2f}x, eficiencia {eficiencia:5.1f}%", file=sys.stderr)
''')
with open('_scan_bench.py', 'w') as f:
    f.write(scan_script)

proc = subprocess.run([sys.executable, '_scan_bench.py'], capture_output=True, text=True, timeout=600)
print(proc.stderr)
if proc.returncode != 0:
    print(proc.stdout)
    raise RuntimeError(f"El script externo fallo con codigo {proc.returncode}")

# Recolectar las lineas RESULT|... en el diccionario compartido de este cuaderno
for line in proc.stdout.splitlines():
    if line.startswith("RESULT|"):
        _, tag, val = line.split("|")
        if tag == "serie":
            resultados[f'rejilla {len(grid)} puntos — serie (1 nucleo)'] = float(val)
        else:
            n = tag.replace("pool", "")
            resultados[f'rejilla {len(grid)} puntos — Pool({n} procesos)'] = float(val)

Serie: 27.86 s total, 278.6 ms/punto
Pool( 2 procesos):  17.94 s total,  179.4 ms/punto, aceleracion 1.55x, eficiencia  77.7%
Pool( 4 procesos):  11.37 s total,  113.7 ms/punto, aceleracion 2.45x, eficiencia  61.3%
Pool(12 procesos):   9.05 s total,   90.5 ms/punto, aceleracion 3.08x, eficiencia  25.7%



La aceleración crece con el número de procesos, pero la **eficiencia**
—aceleración dividida entre número de procesos, 100% sería "cada núcleo
aporta un núcleo entero de trabajo útil"— cae con fuerza según se agregan
más procesos. Dos efectos compiten aquí, y esta rejilla de sólo 100 puntos no
alcanza a separarlos limpiamente:

1. **El costo de arranque por proceso** que la Sección 1 acaba de medir por
   separado: bajo `forkserver` (el método activo en esta máquina, reportado
   en la Sección 0), cada proceso hijo nuevo paga una fracción de ese costo
   de inicialización de Numba antes de poder resolver un solo punto de la
   rejilla — y con sólo ~8-10 puntos por proceso en el reparto de arriba, ese
   costo fijo es una fracción nada despreciable del trabajo total de cada
   proceso.
2. **Contención de CPU**: esta máquina no es necesariamente un servidor
   dedicado con `mp.cpu_count()` núcleos físicos libres en su totalidad — es,
   con toda probabilidad, una máquina compartida o virtualizada (la Sesión 5
   ya usó esa misma expresión), y lanzar más procesos de los que hay
   capacidad física real de por sí produce justo este patrón de eficiencia
   decreciente.

Separar limpiamente cuánto le corresponde a cada efecto —sin poder abrir la
máquina física— queda como pregunta abierta — ver la Sección 11 de `sesion08.md` para la discusión completa.

## 5. Extrapolando al escaneo 5D de referencia

Usando la configuración con mejor tiempo por punto medido arriba, y la misma
ancla de $20^5=3{,}200{,}000$ puntos que las Sesiones 5-7 vienen usando:

In [6]:
mejor_config = min(
    ((k, v) for k, v in resultados.items() if k.startswith(f'rejilla {len(grid)} puntos')),
    key=lambda kv: kv[1]
)
ms_por_punto_paralelo = 1e3 * mejor_config[1] / len(grid)
ms_por_punto_un_nucleo = 1e3 * resultados[f'rejilla {len(grid)} puntos — serie (1 nucleo)'] / len(grid)

N_5D = 20**5
t_5d_un_nucleo = N_5D * ms_por_punto_un_nucleo / 1e3 / 86400  # dias
t_5d_paralelo = N_5D * ms_por_punto_paralelo / 1e3 / 86400    # dias

print(f"Mejor configuracion medida: {mejor_config[0]}  ({ms_por_punto_paralelo:.1f} ms/punto)")
print(f"Escaneo 5D (20^5 = {N_5D:,} puntos):")
print(f"  1 nucleo (Sesion 7, Numba GL+njit, extrapolado aqui): {t_5d_un_nucleo:6.2f} dias")
print(f"  {mp.cpu_count()} procesos (esta sesion):                         {t_5d_paralelo:6.2f} dias  "
      f"({t_5d_un_nucleo/t_5d_paralelo:.2f}x sobre 1 nucleo)")

Mejor configuracion medida: rejilla 100 puntos — Pool(12 procesos)  (90.5 ms/punto)
Escaneo 5D (20^5 = 3,200,000 puntos):
  1 nucleo (Sesion 7, Numba GL+njit, extrapolado aqui):  10.32 dias
  12 procesos (esta sesion):                           3.35 dias  (3.08x sobre 1 nucleo)


## 6. Resumen de lo medido

In [7]:
ancho = max(len(k) for k in resultados)
print(f"{'Experimento':<{ancho}}  {'t promedio':>14}")
print('-' * (ancho + 16))
for nombre, t in resultados.items():
    if t < 1e-3: texto = f"{t*1e6:9.1f} µs"
    elif t < 1.0: texto = f"{t*1e3:9.2f} ms"
    else: texto = f"{t:9.3f} s "
    print(f"{nombre:<{ancho}}  {texto:>14}")

Experimento                                 t promedio
------------------------------------------------------
un punto — solve() individual                310.52 ms
rejilla 100 puntos — serie (1 nucleo)        27.861 s 
rejilla 100 puntos — Pool(2 procesos)        17.939 s 
rejilla 100 puntos — Pool(4 procesos)        11.371 s 
rejilla 100 puntos — Pool(12 procesos)        9.049 s 


## 7. Para llevar

* La pregunta abierta de la Sesión 7 tiene una respuesta medible, no sólo
  conceptual: bajo `fork`, un proceso hijo hereda el binario ya compilado de
  Numba sin costo adicional; bajo `spawn`/`forkserver` —el método que esta
  máquina usa por defecto— cada proceso nuevo paga el costo de que Numba se
  inicialice de nuevo, y `cache=True` sólo evita la recompilación en sí, no
  ese costo de arranque.
* `multiprocessing.Pool.map` reparte un problema vergonzosamente paralelo sin
  escribir sincronización a mano — pero exige que la función de trabajo y
  sus argumentos sean serializables (`pickle`): funciones de nivel de módulo,
  no métodos ligados a una instancia ni `lambda` con estado capturado.
* La eficiencia paralela medida cae con el número de procesos — no es un
  defecto del código de esta sesión, es lo que ocurre cuando el costo fijo
  por-proceso (Sección 1) y la contención de una máquina compartida (Sección
  4) compiten contra una rejilla todavía pequeña. Una rejilla más grande, o
  un `chunksize` mayor en `Pool.map`, amortiguan el primer efecto; el
  segundo depende del hardware real disponible, no del código.
* Lo que sigue sin resolverse — igual que en el cuaderno anterior — es que
  todo esto sigue viviendo dentro de **una sola máquina**. Los núcleos de
  `multiprocessing` comparten memoria y sistema de archivos; escalar más
  allá de una máquina exige un modelo distinto, de paso de mensajes entre
  procesos que ni siquiera comparten memoria — el tema de **MPI**, en la
  Sesión 9.